# YogaVision — Dataset Exploration

Exploratory analysis of the yoga pose dataset and the skeletonization step.

Dataset: [niharika41298/yoga-poses-dataset](https://www.kaggle.com/datasets/niharika41298/yoga-poses-dataset)
Paper: [DOI 10.1007/s12652-022-03910-0](https://doi.org/10.1007/s12652-022-03910-0)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from yogavision.config import Settings
from yogavision.pose_detection import PoseDetector

settings = Settings(Path.cwd().parents[0] / "configs" / "train.yaml")
print("image size:", settings.image_size)
print("classes:", settings.config["data"]["class_names"])

## Class distribution

In [ ]:
root = Path.cwd().parents[0] / "data" / "raw" / "DATASET" / "TRAIN"
counts = {}
for cls_dir in sorted(p for p in root.iterdir() if p.is_dir()):
    n = len([q for q in cls_dir.glob("*.jpg")])
    counts[cls_dir.name] = n

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.keys(), counts.values(), color="#38bdf8")
ax.set_ylabel("images")
ax.set_title("Images per class (TRAIN)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

imbalance = max(counts.values()) / min(counts.values())
print(f"total: {sum(counts.values())}, imbalance ratio: {imbalance:.2f}")

## Original photo vs. skeleton

The paper trains on skeletonized images. Below, the same photo is shown raw
and after MediaPipe skeletonization.

In [ ]:
from yogavision.preprocessing import load_image
from yogavision.skeletonization import skeletonize_image

sample = sorted((root / "tree").glob("*.jpg"))[0]
image = load_image(sample)
skeleton, detected = skeletonize_image(image)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original — {sample.name}")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(skeleton, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Skeleton (detected={detected})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Detection rate

What fraction of the dataset does MediaPipe find a pose in? Images where no
pose is found become all-black canvases.

In [ ]:
from yogavision.pose_detection import PoseDetector

detector = PoseDetector(settings)
found = 0
total = 0
for cls_dir in sorted(p for p in root.iterdir() if p.is_dir()):
    for img_path in sorted(cls_dir.glob("*.jpg"))[:30]:
        result = detector.detect(load_image(img_path))
        total += 1
        found += int(result.detected)
detector.close()

print(f"pose detected in {found}/{total} images ({found / total:.1%})")
print(f"all-black canvases: {total - found} ({(total - found) / total:.1%})")

## Joint angles

Rule-based feedback is computed from these angles. They are only available
when all three contributing landmarks are visible.

In [ ]:
from yogavision.feedback import joint_angles, analyse_feedback

detector = PoseDetector(settings)
for cls in ["tree", "warrior2", "downdog"]:
    img_path = sorted((root / cls).glob("*.jpg"))[0]
    result = detector.detect(load_image(img_path))
    print(f"\n{cls}:")
    print("  angles:", joint_angles(result))
    report = analyse_feedback(result, settings)
    for item in report.items:
        print(f"  [{item.severity}] {item.message}")
detector.close()

## Notes

- 153 of 1550 images in the full dataset produce no detectable pose.
- 179 duplicate image groups exist (MD5), so effective sample count is lower.
- Class imbalance ratio is 1.66.
- These findings are reproduced by `scripts/inspect_dataset.py` and recorded in
  `docs/research_notes.md` and `docs/limitations.md`.